# Credit scenarios

Spread widening via **par CDS / hazard** bumps (uniform and id-specific), **`instrument_price_pct_by_attr`** for sector-style shocks on tagged instruments, and a short note on **rating migration** as a conceptual step beyond parallel spreads.

## Concept

- **Uniform widening**: parallel **`curve_parallel_bp`** with `curve_kind: par_cds` shifts quoted spreads, which the engine maps into **hazard** space (with a discount curve anchor).
- **Sector / name differentiation**: bump one hazard id more than another, or target **`instrument_price_pct_by_attr`** when instruments carry metadata (e.g. `sector=Energy`).
- **Rating migration** (conceptual): a notch downgrade implies a **new spread curve** or **higher hazard**, often larger than a parallel +X bp stress—typically modeled as a **regime change**, not a tiny bump.

We compare **survival** \(S(t)\) at selected horizons before and after shocks.

In [ ]:
import json

from finstack.scenarios import (
    list_builtin_templates,
    build_from_template,
    list_template_components,
    build_template_component,
    build_scenario_spec,
    compose_scenarios,
    apply_scenario_to_market,
    validate_scenario_spec,
)
print(
    "Imports OK",
    callable(compose_scenarios),
    callable(apply_scenario_to_market),
    callable(build_template_component),
)

print("Templates available:", list_builtin_templates())

try:
    tmeta = build_from_template("covid_2020")
    print("Sample historical template covid_2020 (credit excerpt search):")
    print(tmeta[:400], "...")
except Exception as e:
    print("Could not load covid_2020 template:", e)

# Uniform widening on a single issuer hazard
ops_uniform = [
    {
        "kind": "curve_parallel_bp",
        "curve_kind": "par_cds",
        "curve_id": "CORP-IG",
        "discount_curve_id": "USD-OIS",
        "bp": 75.0,
    }
]
su = build_scenario_spec("credit_uniform", json.dumps(ops_uniform), "Uniform +75bp", "All buckets widen")
print("validate uniform:", validate_scenario_spec(su))

# Sector-style: widen HY more than IG (two curve IDs)
ops_sector = [
    {
        "kind": "curve_parallel_bp",
        "curve_kind": "par_cds",
        "curve_id": "CORP-IG",
        "discount_curve_id": "USD-OIS",
        "bp": 25.0,
    },
    {
        "kind": "curve_parallel_bp",
        "curve_kind": "par_cds",
        "curve_id": "CORP-HY",
        "discount_curve_id": "USD-OIS",
        "bp": 150.0,
    },
]
ss = build_scenario_spec("credit_sector", json.dumps(ops_sector), "IG vs HY", "HY wider")
print("validate sector:", validate_scenario_spec(ss))

# Metadata-based price shock (wildcard attrs = all instruments with meta — here illustrative JSON only)
ops_attr = [
    {
        "kind": "instrument_price_pct_by_attr",
        "attrs": {"sector": "Energy"},
        "pct": -2.5,
    }
]
sa = build_scenario_spec("energy_px", json.dumps(ops_attr), "Energy price", "-2.5% on Energy-tagged instruments")
print("validate attr shock:", validate_scenario_spec(sa))
print("Built three credit-related scenario specs (uniform, id-specific, attribute).")


In [ ]:
import json
from datetime import date

from finstack.core.market_data import DiscountCurve, HazardCurve, MarketContext
from finstack.scenarios import apply_scenario_to_market, build_scenario_spec

AS_OF = "2025-01-15"
bd = date(2025, 1, 15)

usd = DiscountCurve("USD-OIS", bd, [(0.0, 1.0), (1.0, 0.98), (5.0, 0.90)], day_count="act_365f")
ig = HazardCurve(
    "CORP-IG",
    bd,
    [(0.0, 0.0), (1.0, 0.012), (3.0, 0.018), (5.0, 0.022)],
    recovery_rate=0.4,
)
hy = HazardCurve(
    "CORP-HY",
    bd,
    [(0.0, 0.0), (1.0, 0.045), (3.0, 0.055), (5.0, 0.06)],
    recovery_rate=0.35,
)

mc = MarketContext().insert(usd).insert(ig).insert(hy)
base_json = mc.to_json()

base_ig = mc.get_hazard("CORP-IG")
base_hy = mc.get_hazard("CORP-HY")
for t in (1.0, 3.0, 5.0):
    print(f"Base survival CORP-IG t={t}Y S={base_ig.survival(t):.6f}  HY S={base_hy.survival(t):.6f}")


def hazard_from_stressed(mj: str, hid: str):
    for c in json.loads(mj).get("curves", []):
        if c.get("type") == "hazard" and c.get("id") == hid:
            knots = [(float(x), float(y)) for x, y in c["knot_points"]]
            return HazardCurve(
                c["id"],
                date.fromisoformat(c["base"]),
                knots,
                recovery_rate=float(c.get("recovery_rate", 0.4)),
                day_count=c.get("day_count", "act_365f"),
            )
    raise ValueError(hid)


def apply_and_print(label: str, spec_json: str):
    r = apply_scenario_to_market(spec_json, base_json, AS_OF)
    print(f"\n--- {label} ---")
    print("  operations_applied:", r["operations_applied"], " warnings:", r["warnings"])
    mj = r["market_json"]
    h2 = hazard_from_stressed(mj, "CORP-IG")
    h3 = hazard_from_stressed(mj, "CORP-HY")
    for t in (1.0, 3.0, 5.0):
        print(
            f"  t={t}Y  IG S {base_ig.survival(t):.6f} -> {h2.survival(t):.6f}  |  HY {base_hy.survival(t):.6f} -> {h3.survival(t):.6f}"
        )


spec_uniform = build_scenario_spec(
    "u",
    json.dumps(
        [
            {
                "kind": "curve_parallel_bp",
                "curve_kind": "par_cds",
                "curve_id": "CORP-IG",
                "discount_curve_id": "USD-OIS",
                "bp": 80.0,
            }
        ]
    ),
    "n",
    "d",
)

spec_both = build_scenario_spec(
    "b",
    json.dumps(
        [
            {
                "kind": "curve_parallel_bp",
                "curve_kind": "par_cds",
                "curve_id": "CORP-IG",
                "discount_curve_id": "USD-OIS",
                "bp": 30.0,
            },
            {
                "kind": "curve_parallel_bp",
                "curve_kind": "par_cds",
                "curve_id": "CORP-HY",
                "discount_curve_id": "USD-OIS",
                "bp": 120.0,
            },
        ]
    ),
    "n",
    "d",
)

apply_and_print("Uniform IG +80bp par CDS shock", spec_uniform)
apply_and_print("Sector-style IG +30bp, HY +120bp", spec_both)
print("\nRating migration (conceptual): a multi-notch move usually implies re-calibrating to a new curve level/shape, not a single parallel bump.")


## Takeaways

- **`par_cds`** shocks are the primary way to stress **credit curves** tied to **`HazardCurve`** ids; supply **`discount_curve_id`** when the engine needs a rate anchor.
- **Uniform vs differentiated** credit stress is expressed by **one shared bump** or **different `curve_id` targets** (and magnitudes).
- **`instrument_price_pct_by_attr`** is useful when positions carry **metadata** you want to shock as a group.
- **Survival** drops when spreads widen—rebuild **`HazardCurve`** from the stressed **`market_json`** to print \(S(t)\) after the scenario.